# Setup

In [ ]:
!pip -q install pandas matplotlib tqdm scikit-learn opencv-python-headless

# Imports

In [ ]:
import os, re, math, random, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# Data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from google.colab import files
import zipfile
uploaded = files.upload()
if uploaded:
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall("/content/datasets")

In [ ]:
DATA_ROOT = "/content/datasets/Full dataset"
LABELS_FILE = os.path.join(DATA_ROOT, "labels.txt")
IMAGES_DIR = os.path.join(DATA_ROOT, "images")
assert os.path.exists(LABELS_FILE)

In [ ]:
df = pd.read_csv(LABELS_FILE, sep="	", header=None, names=["rel_path", "text"])
df["text"] = df["text"].astype(str).apply(lambda s: unicodedata.normalize("NFKC", s))
df["text"] = df["text"].apply(lambda s: re.sub(r"\s+", " ", s).strip())
df["image_path"] = df["rel_path"].apply(lambda p: os.path.join(DATA_ROOT, p))
df = df[df["text"].str.len() > 0].reset_index(drop=True)
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
len(df)

# Statistics

In [ ]:
df["text_len"] = df["text"].str.len()
df["text_len"].describe()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df["text_len"], bins=30)
plt.title("Text length")
plt.xlabel("Length")
plt.ylabel("Count")
plt.show()

In [ ]:
sample_df = df.sample(min(6, len(df)), random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes = axes.flatten()
for ax, (_, row) in zip(axes, sample_df.iterrows()):
    img = Image.open(row["image_path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(row["text"][:30])
    ax.axis("off")
plt.tight_layout()

# Charset

In [ ]:
alphabet = sorted(set("".join(df["text"].tolist())))
char_to_idx = {c: i + 1 for i, c in enumerate(alphabet)}
idx_to_char = {i + 1: c for i, c in enumerate(alphabet)}
len(alphabet)

# Dataloaders

In [ ]:
IMG_HEIGHT = 32
MAX_WIDTH = 512
def preprocess_image(img, augment=False):
    img = ImageOps.grayscale(img)
    if augment:
        if random.random() < 0.4:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
        if random.random() < 0.4:
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
        if random.random() < 0.25:
            img = img.rotate(random.uniform(-2.0, 2.0), expand=False, fillcolor=255)
        if random.random() < 0.2:
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.0, 1.0)))
    w, h = img.size
    new_w = int(IMG_HEIGHT * w / h)
    new_w = max(1, min(new_w, MAX_WIDTH))
    img = img.resize((new_w, IMG_HEIGHT), Image.BILINEAR)
    arr = np.asarray(img).astype(np.float32) / 255.0
    tensor = torch.from_numpy(arr).unsqueeze(0)
    return tensor
def encode_text(text):
    return [char_to_idx[c] for c in text if c in char_to_idx]
class OCRDataset(Dataset):
    def __init__(self, data_frame, augment=False):
        self.df = data_frame.reset_index(drop=True)
        self.augment = augment
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"])
        tensor = preprocess_image(img, augment=self.augment)
        text = row["text"]
        return tensor, text
def collate_fn(batch):
    images, texts = zip(*batch)
    widths = [img.shape[2] for img in images]
    max_w = max(widths)
    batch_size = len(images)
    padded = torch.zeros(batch_size, 1, IMG_HEIGHT, max_w, dtype=torch.float32)
    for i, img in enumerate(images):
        w = img.shape[2]
        padded[i, :, :, :w] = img
    targets = []
    target_lengths = []
    for text in texts:
        encoded = encode_text(text)
        targets.extend(encoded)
        target_lengths.append(len(encoded))
    targets = torch.tensor(targets, dtype=torch.long)
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)
    widths = torch.tensor(widths, dtype=torch.long)
    return padded, targets, target_lengths, widths, list(texts)

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.1, random_state=SEED, shuffle=True)
train_df, val_df = train_test_split(train_df, test_size=0.1111, random_state=SEED, shuffle=True)
train_ds = OCRDataset(train_df, augment=True)
val_ds = OCRDataset(val_df, augment=False)
test_ds = OCRDataset(test_df, augment=False)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, collate_fn=collate_fn)
len(train_ds), len(val_ds), len(test_ds)

# Model

In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.time_downsample = 4
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1),
            nn.ReLU(True),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(256, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.Conv2d(512, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(512, 512, 2, 1, 0),
            nn.ReLU(True)
        )
        self.rnn = nn.LSTM(512, 256, num_layers=2, bidirectional=True, dropout=0.1)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        x = self.cnn(x)
        x = x.squeeze(2)
        x = x.permute(2, 0, 1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        x = nn.functional.log_softmax(x, dim=2)
        return x

# Training

In [ ]:
use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)
def greedy_decode(log_probs, idx_to_char, blank=0):
    max_indices = log_probs.argmax(2)
    results = []
    for n in range(max_indices.shape[1]):
        seq = max_indices[:, n].tolist()
        prev = None
        decoded = []
        for idx in seq:
            if idx != blank and idx != prev:
                decoded.append(idx_to_char.get(idx, ""))
            prev = idx
        results.append("".join(decoded))
    return results
def edit_distance(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )
    return dp[-1][-1]
def cer(pred, truth):
    if len(truth) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    return edit_distance(pred, truth) / len(truth)
def wer(pred, truth):
    pred_words = pred.split()
    truth_words = truth.split()
    if len(truth_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    return edit_distance(pred_words, truth_words) / len(truth_words)
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for images, targets, target_lengths, widths, _ in tqdm(loader):
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        widths = widths.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            log_probs = model(images)
            T = log_probs.size(0)
            input_lengths = torch.clamp(widths // model.time_downsample, min=1, max=T)
            loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / max(1, len(loader))
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_cer = 0.0
    total_wer = 0.0
    total_acc = 0.0
    total_samples = 0
    for images, targets, target_lengths, widths, texts in loader:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        widths = widths.to(device)
        log_probs = model(images)
        T = log_probs.size(0)
        input_lengths = torch.clamp(widths // model.time_downsample, min=1, max=T)
        loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
        total_loss += loss.item()
        preds = greedy_decode(log_probs.detach().cpu(), idx_to_char, blank=0)
        for pred, truth in zip(preds, texts):
            total_cer += cer(pred, truth)
            total_wer += wer(pred, truth)
            total_acc += 1.0 if pred == truth else 0.0
            total_samples += 1
    avg_loss = total_loss / max(1, len(loader))
    avg_cer = total_cer / max(1, total_samples)
    avg_wer = total_wer / max(1, total_samples)
    avg_acc = total_acc / max(1, total_samples)
    return avg_loss, avg_cer, avg_wer, avg_acc

In [ ]:
num_classes = len(alphabet) + 1
model = CRNN(num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-5)
EPOCHS = 30
history = {"train_loss": [], "val_loss": [], "val_cer": [], "val_wer": [], "val_acc": []}
best_cer = 1e9
best_path = "best_darija_crnn.pth"
early_stop_patience = 6
bad_epochs = 0
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_cer, val_wer, val_acc = evaluate(model, val_loader)
    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_cer"].append(val_cer)
    history["val_wer"].append(val_wer)
    history["val_acc"].append(val_acc)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_cer={val_cer:.4f} | val_wer={val_wer:.4f} | val_acc={val_acc:.4f}")
    if val_cer + 1e-6 < best_cer:
        best_cer = val_cer
        bad_epochs = 0
        torch.save({"model_state": model.state_dict(), "alphabet": alphabet, "img_height": IMG_HEIGHT, "max_width": MAX_WIDTH, "epoch": epoch, "val_cer": val_cer}, best_path)
    else:
        bad_epochs += 1
        if bad_epochs >= early_stop_patience:
            break

# Curves

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.title("Loss")
plt.legend()
plt.subplot(1, 3, 2)
plt.plot(history["val_cer"], label="val_cer")
plt.plot(history["val_wer"], label="val_wer")
plt.title("CER/WER")
plt.legend()
plt.subplot(1, 3, 3)
plt.plot(history["val_acc"], label="val_acc")
plt.title("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# Test

In [ ]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
test_loss, test_cer, test_wer, test_acc = evaluate(model, test_loader)
test_loss, test_cer, test_wer, test_acc

# Samples

In [ ]:
@torch.no_grad()
def predict_batch(model, images):
    log_probs = model(images.to(device))
    preds = greedy_decode(log_probs.detach().cpu(), idx_to_char, blank=0)
    return preds
sample_batch = next(iter(test_loader))
images, _, _, _, texts = sample_batch
preds = predict_batch(model, images)
for i in range(min(8, len(preds))):
    print(texts[i])
    print(preds[i])
    print()

# Save

In [ ]:
last_path = "darija_crnn_last.pth"
torch.save({"model_state": model.state_dict(), "alphabet": alphabet, "img_height": IMG_HEIGHT, "max_width": MAX_WIDTH}, last_path)
last_path

# Load

In [ ]:
def load_model(model_path):
    obj = torch.load(model_path, map_location=device)
    chars = obj.get("alphabet") or obj.get("charset") or obj.get("chars") or alphabet
    if isinstance(chars, str):
        chars = list(chars)
    idx_to_char_local = {i + 1: c for i, c in enumerate(chars)}
    model_local = CRNN(len(chars) + 1).to(device)
    model_local.load_state_dict(obj["model_state"])
    model_local.eval()
    cfg = {"img_height": obj.get("img_height", IMG_HEIGHT), "max_width": obj.get("max_width", MAX_WIDTH)}
    return model_local, idx_to_char_local, cfg
model_loaded, idx_to_char_loaded, cfg = load_model(best_path)

# Predict Image

In [ ]:
def preprocess_image_for_infer(img, img_height, max_width):
    img = ImageOps.grayscale(img)
    w, h = img.size
    new_w = int(img_height * w / h)
    new_w = max(1, min(new_w, max_width))
    img = img.resize((new_w, img_height), Image.BILINEAR)
    arr = np.asarray(img).astype(np.float32) / 255.0
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
    return tensor
@torch.no_grad()
def predict_image_path(model, image_path, idx_to_char_local, img_height, max_width):
    img = Image.open(image_path)
    tensor = preprocess_image_for_infer(img, img_height, max_width)
    log_probs = model(tensor.to(device))
    pred = greedy_decode(log_probs.detach().cpu(), idx_to_char_local, blank=0)[0]
    return pred
uploaded = files.upload()
if uploaded:
    test_img = next(iter(uploaded))
    predict_image_path(model_loaded, test_img, idx_to_char_loaded, cfg["img_height"], cfg["max_width"])